# 1. Install necessary Libraries

In this part, we'll begin by installing necessary libraries needed for running our computer vision training and testing scripts

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install transformers scikit-learn pillow pandas numpy huggingface_hub ipywidgets

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Import necessary libraries
After installing the dependencies, we will begin importing the libraries that are going to be used

In [2]:
import numpy as np
import pandas as pd
import json
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import ViTForImageClassification
from transformers import AutoImageProcessor
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score, precision_score, f1_score
from PIL import Image
import os

from huggingface_hub import notebook_login
notebook_login()

# 3. Custom Dataset Loader
Since we're using a custom dataset, we will need to write a custom dataset loader to pass our image data to the model


In [3]:
class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.data = df # Differentiate between train_df, val_df and test_df
        self.img_dir = img_dir # Image Directory Path
        self.transform = transform # Data augmentation

    def __len__(self):
        return len(self.data) # Calculate the number of rows of the dataset

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image'] # Get the image name from the csv header. Used for path later
        label = int(self.data.iloc[idx]['Label']) # Get the Label value from csv header

        img_path = os.path.join(self.img_dir, img_name) # Combine the root image dir with the image name to get the full individual image path
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# 4. Setup Cuda Device and load pre-trained model
Here in this part, we check if PyTorch detects our GPU cuda cores, then load the pre-trained computer vision model if true

In [4]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Cuda available: ", torch.cuda.is_available())

processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')

Cuda available:  True


# 5. Data Preprocessing
In this part, we will perform data augmentation on our dataset, then split our dataset into training and validation data

In [5]:
# Data augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

df = pd.read_csv('dataset/labels.csv')
labels = df['Label']
train_df, val_df = train_test_split(df, test_size=0.2, stratify=labels, random_state=42) #Normal train_test_split

train_dataset = MedicalDatasetLoader(train_df, "dataset/image/", train_transform)
val_dataset = MedicalDatasetLoader(val_df, "dataset/image/", val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
)

# 6. Setup Model
In this part, we will design the inner architecture of ViT such as its layers, activation function types and Feed Forward Network (FFN) layers

In [6]:
# Load Google ViT model
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=5, # Number of labels type in our dataset
    ignore_mismatched_sizes=True
).to(device)

You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([5, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


# 7. Hyperparameters tuning
We set up our class weights, epochs number, early stopping mechanism, optimizer and learning rate scheduler to optimize our model performance and provide better generalization through reducing overfitting

In [7]:
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['Label']), y=train_df['Label'])
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

num_epochs = 20
early_stopping_patience = 5
epochs_no_improvement = 0
best_val_loss = float('inf')
early_stopping = False

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
scaler = GradScaler()

# 8. Experiment Logging Class

In [8]:
class ExperimentTracker:
    def __init__(self, base_dir="experiments"):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join(base_dir, f"exp_{timestamp}")
        os.makedirs(self.exp_dir)

        self.epoch_metrics = []
        self.final_metrics = {}
        self.config = {}

    # ---------------- CONFIG ----------------
    def log_config(self, config):
        self.config = config
        with open(os.path.join(self.exp_dir, "config.json"), "w") as f:
            json.dump(config, f, indent=4)

    # ---------------- PER EPOCH ----------------
    def log_epoch(self, epoch, train_loss, val_loss, train_acc, val_acc):
        self.epoch_metrics.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

    # ---------------- FINAL METRICS ----------------
    def log_final_metrics(self, acc, prec, rec, f1, cm):
        self.final_metrics = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "confusion_matrix": cm.tolist()  # convert numpy → JSON
        }

    # ---------------- SAVE ----------------
    def save_all(self):
        with open(os.path.join(self.exp_dir, "metrics.json"), "w") as f:
            json.dump({
                "config": self.config,
                "epoch_metrics": self.epoch_metrics,
                "final_metrics": self.final_metrics
            }, f, indent=4)

    def save_model(self, model, name="best_model.pth"):
        torch.save(model.state_dict(), os.path.join(self.exp_dir, name))

# 9. Training and Validation Script
In this part, we train and test our model and finally save the final weights of the model

In [9]:
CONFIG = {
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "epochs": num_epochs,
    "batch_size": 16,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR"
}

tracker = ExperimentTracker()
tracker.log_config(CONFIG)

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            outputs = model(pixel_values=inputs)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.logits, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        with autocast(device_type='cuda'):
            outputs = model(pixel_values=inputs)
            loss = criterion(outputs.logits, labels)

        probs = F.softmax(outputs.logits, dim=1)

        val_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.logits, 1)

        val_total += inputs.size(0)
        val_correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    # ---------------- METRICS ----------------
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    scheduler.step()

    # ---------------- TRACKING ----------------
    tracker.log_epoch(epoch, epoch_train_loss, epoch_val_loss, epoch_train_acc, epoch_val_acc)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improvement = 0
        tracker.save_model(model)
    else:
        epochs_no_improvement += 1
        if epochs_no_improvement >= early_stopping_patience:
            print('Early stopping')
            break

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

tracker.log_final_metrics(acc, prec, rec, f1, cm)
tracker.save_all()

Epoch [1/20]
Train Loss: 1.4338 | Train Acc: 0.5816
Val Loss: 1.2192 | Val Acc: 0.7247
Epoch [2/20]
Train Loss: 1.2584 | Train Acc: 0.7067
Val Loss: 1.2000 | Val Acc: 0.7753
Epoch [3/20]
Train Loss: 1.1825 | Train Acc: 0.7375
Val Loss: 1.1472 | Val Acc: 0.7824
Epoch [4/20]
Train Loss: 1.0939 | Train Acc: 0.7874
Val Loss: 1.1300 | Val Acc: 0.7702
Epoch [5/20]
Train Loss: 1.0260 | Train Acc: 0.8216
Val Loss: 1.1427 | Val Acc: 0.7945
Epoch [6/20]
Train Loss: 0.9819 | Train Acc: 0.8438
Val Loss: 1.1587 | Val Acc: 0.7874
Epoch [7/20]
Train Loss: 0.9121 | Train Acc: 0.8778
Val Loss: 1.1602 | Val Acc: 0.7955
Epoch [8/20]
Train Loss: 0.8579 | Train Acc: 0.9005
Val Loss: 1.1759 | Val Acc: 0.8107
Early stopping


TypeError: ExperimentTracker.log_final_metrics() missing 1 required positional argument: 'cm'

# 10. Metrics Visualization
Lastly, in the final parts we'll measure our model performance based on the following metrics:
- Accuracy: Calculate the ratio of correctly predicted labels
- Precision: Calculate how many positive predictions are truly positive (Minimize false positives)
- Recall: Calculate how many postivive labels are predicted (Minimize false negatives)
- F1-Score: Determine the mean of precision and recall combined
- Confusion Matrix: Determine the total True Positive, False Positives, True Negatives and False Negatives that our model made

In [10]:
print(f"Accuracy: {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall: {rec * 100:.2f}%")
print(f"F1-Score: {f1 * 100:.2f}%")
print("Confusion Matrix: ")
print(cm)

Accuracy: 82.09%
Precision: 70.14%
Recall: 67.99%
F1-Score: 68.97%
Confusion Matrix: 
[[437   9   2   0   0]
 [ 17  58  27   1   2]
 [ 11  24 215  16  11]
 [  0   1  21  15   7]
 [  0   1  19   8  86]]
